# HQ200: structure and example readers

This notebook inventories the five ARKit capture scenes and reads representative RGB, depth, confidence, camera-pose, mesh, material, annotation, and world-map files. It does not modify the dataset.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


In [ ]:
# Override this line if the project is moved.
PROJECT_ROOT = Path(r"C:\Users\liter\Documents\ITU_MSc\Project_Thesis")
HQ200_ROOT = PROJECT_ROOT / "data" / "HQ200"

if not HQ200_ROOT.is_dir():
    # Also support a kernel whose working directory is the project or ResearchProject folder.
    candidates = [Path.cwd() / "data" / "HQ200", Path.cwd().parent / "data" / "HQ200"]
    HQ200_ROOT = next((p.resolve() for p in candidates if p.is_dir()), HQ200_ROOT)

if not HQ200_ROOT.is_dir():
    raise FileNotFoundError(f"HQ200 directory not found: {HQ200_ROOT}")

SCENES = sorted(p for p in HQ200_ROOT.iterdir() if p.is_dir())
print(f"HQ200 root: {HQ200_ROOT}")
print(f"Scenes: {len(SCENES)}")


## 1. Dataset inventory

In [ ]:
def scene_inventory(scene):
    files = [p for p in scene.iterdir() if p.is_file()]
    return {
        "scene": scene.name,
        "rgb_frames": len(list(scene.glob("frame_*.jpg"))),
        "camera_json": len(list(scene.glob("frame_*.json"))),
        "depth_maps": len(list(scene.glob("depth_*.png"))),
        "confidence_maps": len(list(scene.glob("conf_*.png"))),
        "obj_meshes": len(list(scene.glob("*.obj"))),
        "arkit_maps": len(list(scene.glob("*.arkit"))),
        "files_total": len(files),
        "size_mb": round(sum(p.stat().st_size for p in files) / 1024**2, 1),
    }

inventory = pd.DataFrame(scene_inventory(scene) for scene in SCENES)
display(inventory)
print(f"Total size: {inventory['size_mb'].sum() / 1024:.3f} GB")


## 2. Select a scene and aligned frame

In [ ]:
SCENE_INDEX = 0
FRAME_INDEX = 0

scene = SCENES[SCENE_INDEX]
stem = f"{FRAME_INDEX:05d}"
rgb_path = scene / f"frame_{stem}.jpg"
depth_path = scene / f"depth_{stem}.png"
confidence_path = scene / f"conf_{stem}.png"
camera_path = scene / f"frame_{stem}.json"

print("Scene:", scene.name)
for label, path in {"RGB": rgb_path, "depth": depth_path, "confidence": confidence_path, "camera": camera_path}.items():
    print(f"{label:10s} {path.name:22s} exists={path.is_file()}")

# RGB images are sampled less frequently than depth/camera frames.
if not rgb_path.is_file():
    rgb_path = min(scene.glob("frame_*.jpg"), key=lambda p: abs(int(p.stem.split('_')[1]) - FRAME_INDEX))
    print("Nearest available RGB frame:", rgb_path.name)


In [ ]:
rgb = np.asarray(Image.open(rgb_path).convert("RGB"))
depth_raw = np.asarray(Image.open(depth_path))
confidence = np.asarray(Image.open(confidence_path))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(rgb)
axes[0].set_title(f"RGB: {rgb_path.name}\n{rgb.shape[1]} x {rgb.shape[0]}")
depth_view = axes[1].imshow(depth_raw, cmap="turbo")
axes[1].set_title(f"Depth (raw uint16): {depth_path.name}\nrange {depth_raw.min()}-{depth_raw.max()}")
fig.colorbar(depth_view, ax=axes[1], fraction=0.046)
conf_view = axes[2].imshow(confidence, cmap="viridis", vmin=0, vmax=2)
axes[2].set_title(f"Confidence: {confidence_path.name}\nvalues {np.unique(confidence).tolist()}")
fig.colorbar(conf_view, ax=axes[2], ticks=[0, 1, 2], fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Camera metadata

In [ ]:
camera = json.loads(camera_path.read_text(encoding="utf-8"))
intrinsics = np.asarray(camera["intrinsics"], dtype=float).reshape(3, 3)
camera_pose = np.asarray(camera["cameraPoseARFrame"], dtype=float).reshape(4, 4)
projection = np.asarray(camera["projectionMatrix"], dtype=float).reshape(4, 4)

print("Available fields:", sorted(camera))
print("Frame index:", camera.get("frame_index"))
print("Time:", camera.get("time"))
print("Motion quality:", camera.get("motionQuality"))
print("Intrinsics:\n", intrinsics)
print("Camera pose (AR frame):\n", camera_pose)
print("Projection matrix:\n", projection)


In [ ]:
# Visualize the complete ARKit camera trajectory and a subset of camera frustums.
# cameraPoseARFrame is treated as camera-to-world; ARKit cameras look along local -Z.
camera_files = sorted(scene.glob("frame_*.json"))
poses = []
frame_ids = []
for path in camera_files:
    metadata = json.loads(path.read_text(encoding="utf-8"))
    poses.append(np.asarray(metadata["cameraPoseARFrame"], dtype=float).reshape(4, 4))
    frame_ids.append(int(metadata.get("frame_index", path.stem.split("_")[-1])))
poses = np.stack(poses)
centers = poses[:, :3, 3]

fig = plt.figure(figsize=(11, 9))
ax = fig.add_subplot(111, projection="3d")

# Optional raw-mesh overlay. OBJ vertices form a point set as well as mesh vertices.
mesh_path = scene / "export.obj"
mesh_vertices = []
with mesh_path.open(encoding="utf-8", errors="replace") as handle:
    for line in handle:
        if line.startswith("v "):
            mesh_vertices.append([float(value) for value in line.split()[1:4]])
mesh_vertices = np.asarray(mesh_vertices)
mesh_step = max(1, len(mesh_vertices) // 10000)
ax.scatter(*mesh_vertices[::mesh_step].T, s=0.4, c="lightgray", alpha=0.25, label="export.obj vertices")

trajectory = ax.scatter(*centers.T, c=frame_ids, cmap="viridis", s=6, label="camera centers")
ax.plot(*centers.T, color="black", linewidth=0.5, alpha=0.5)

# Draw about 30 frustums. R columns are camera right, up, and backward axes.
frustum_step = max(1, len(poses) // 30)
frustum_depth = max(np.ptp(centers, axis=0).max() * 0.035, 0.08)
for transform in poses[::frustum_step]:
    rotation = transform[:3, :3]
    center = transform[:3, 3]
    right, up, backward = rotation[:, 0], rotation[:, 1], rotation[:, 2]
    forward = -backward
    target = center + frustum_depth * forward
    half_width = 0.55 * frustum_depth
    half_height = 0.40 * frustum_depth
    corners = np.array([
        target - half_width * right - half_height * up,
        target + half_width * right - half_height * up,
        target + half_width * right + half_height * up,
        target - half_width * right + half_height * up,
    ])
    for corner in corners:
        ax.plot(*np.vstack([center, corner]).T, color="tab:red", linewidth=0.7, alpha=0.7)
    closed = np.vstack([corners, corners[0]])
    ax.plot(*closed.T, color="tab:red", linewidth=0.7, alpha=0.7)

# Equal scaling prevents a distorted trajectory.
all_points = np.vstack([centers, mesh_vertices])
lower, upper = all_points.min(axis=0), all_points.max(axis=0)
mid = (lower + upper) / 2
radius = (upper - lower).max() / 2
ax.set_xlim(mid[0] - radius, mid[0] + radius)
ax.set_ylim(mid[1] - radius, mid[1] + radius)
ax.set_zlim(mid[2] - radius, mid[2] + radius)
ax.set_xlabel("ARKit X")
ax.set_ylabel("ARKit Y")
ax.set_zlabel("ARKit Z")
ax.set_title(f"ARKit camera poses: {scene.name}")
fig.colorbar(trajectory, ax=ax, shrink=0.65, label="frame index")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


## 4. Mesh, material, annotations, and ARKit world map

In [ ]:
def summarize_obj(path):
    counts = Counter()
    with path.open(encoding="utf-8", errors="replace") as handle:
        for line in handle:
            token = line.split(maxsplit=1)[0] if line.strip() else "blank"
            counts[token] += 1
    return {
        "file": path.name,
        "size_mb": round(path.stat().st_size / 1024**2, 2),
        "vertices": counts["v"],
        "texture_coords": counts["vt"],
        "normals": counts["vn"],
        "faces": counts["f"],
    }

mesh_summary = pd.DataFrame(summarize_obj(path) for path in sorted(scene.glob("*.obj")))
display(mesh_summary)

annotations_path = scene / "annotations.json"
annotations = json.loads(annotations_path.read_text(encoding="utf-8"))
print("Annotations:", annotations)

for material_path in sorted(scene.glob("*.mtl")):
    print(f"\nMaterial file: {material_path.name}")
    print("".join(material_path.read_text(encoding="utf-8", errors="replace").splitlines(True)[:20]))

for world_map_path in sorted(scene.glob("*.arkit")):
    header = world_map_path.read_bytes()[:8]
    print(f"ARKit map: {world_map_path.name}, {world_map_path.stat().st_size / 1024**2:.2f} MB, header={header!r}")
    print("The bplist00 header identifies an Apple binary property-list archive; this notebook reports it without attempting unsafe object deserialization.")


## 5. Browse several RGB examples

In [ ]:
N_EXAMPLES = 8
rgb_paths = sorted(scene.glob("frame_*.jpg"))
sample_positions = np.linspace(0, len(rgb_paths) - 1, min(N_EXAMPLES, len(rgb_paths)), dtype=int)
sample_paths = [rgb_paths[i] for i in sample_positions]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, path in zip(axes.flat, sample_paths):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(path.name)
    ax.axis("off")
for ax in axes.flat[len(sample_paths):]:
    ax.axis("off")
plt.suptitle(scene.name)
plt.tight_layout()
plt.show()
